In [38]:
import os 
from dotenv import load_dotenv

# langchain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_community.embeddings import JinaEmbeddings

In [40]:


groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("ENV VAR LOADED")



ENV VAR LOADED


In [41]:
DATA_FILE_PATH = os.path.join("data" , "microsoft_hr_policy_public_policy_only.txt")


In [42]:
DATA_FILE_PATH

'data\\microsoft_hr_policy_public_policy_only.txt'

In [43]:
loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")
documents = loader.load()

print(f"Loaded file: {DATA_FILE_PATH}")
print(f"Number of documents loaded: {len(documents)}")
print(f"Total characters in document: {len(documents[0].page_content)}")
print("\n--- Preview of first 300 characters ---")
print(documents[0].page_content[:300])


Loaded file: data\microsoft_hr_policy_public_policy_only.txt
Number of documents loaded: 1
Total characters in document: 18114

--- Preview of first 300 characters ---
MICROSOFT HR POLICY ASSISTANT
PUBLICLY AVAILABLE MICROSOFT HR / EMPLOYEE INFORMATION

IMPORTANT:
This dataset contains information publicly available from official Microsoft
websites. It is NOT Microsoft's complete internal HR handbook. Some detailed
employee policies are internal or location-specif


In [44]:
documents

[Document(metadata={'source': 'data\\microsoft_hr_policy_public_policy_only.txt'}, page_content='MICROSOFT HR POLICY ASSISTANT\nPUBLICLY AVAILABLE MICROSOFT HR / EMPLOYEE INFORMATION\n\nIMPORTANT:\nThis dataset contains information publicly available from official Microsoft\nwebsites. It is NOT Microsoft\'s complete internal HR handbook. Some detailed\nemployee policies are internal or location-specific. The assistant must not\ninvent missing policies.\n\n============================================================\n1. COMPANY OVERVIEW\n============================================================\n\nCompany:\nMicrosoft Corporation\n\nMicrosoft\'s mission is to empower every person and every organization on\nthe planet to achieve more.\n\nMicrosoft describes its culture around a growth mindset, customer obsession,\nOne Microsoft, diversity and inclusion, and making a difference.\n\nMicrosoft\'s Standards of Business Conduct / Trust Code emphasizes:\n- Respect\n- Integrity\n- Accountabil

In [45]:
print(documents[0].page_content)

MICROSOFT HR POLICY ASSISTANT
PUBLICLY AVAILABLE MICROSOFT HR / EMPLOYEE INFORMATION

IMPORTANT:
This dataset contains information publicly available from official Microsoft
websites. It is NOT Microsoft's complete internal HR handbook. Some detailed
employee policies are internal or location-specific. The assistant must not
invent missing policies.

1. COMPANY OVERVIEW

Company:
Microsoft Corporation

Microsoft's mission is to empower every person and every organization on
the planet to achieve more.

Microsoft describes its culture around a growth mindset, customer obsession,
One Microsoft, diversity and inclusion, and making a difference.

Microsoft's Standards of Business Conduct / Trust Code emphasizes:
- Respect
- Integrity
- Accountability

SOURCE:
Microsoft Standards of Business Conduct
https://www.microsoft.com/en-us/legal/compliance/sbc/standards


2. EMPLOYEE BENEFITS

Microsoft publicly describes employee benefits that support areas including:

- Physical wellbeing
- Mental

In [46]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(chunks)

[Document(metadata={'source': 'data\\microsoft_hr_policy_public_policy_only.txt'}, page_content="MICROSOFT HR POLICY ASSISTANT\nPUBLICLY AVAILABLE MICROSOFT HR / EMPLOYEE INFORMATION\n\nIMPORTANT:\nThis dataset contains information publicly available from official Microsoft\nwebsites. It is NOT Microsoft's complete internal HR handbook. Some detailed\nemployee policies are internal or location-specific. The assistant must not\ninvent missing policies.\n\n============================================================\n1. COMPANY OVERVIEW\n============================================================"), Document(metadata={'source': 'data\\microsoft_hr_policy_public_policy_only.txt'}, page_content="Company:\nMicrosoft Corporation\n\nMicrosoft's mission is to empower every person and every organization on\nthe planet to achieve more.\n\nMicrosoft describes its culture around a growth mindset, customer obsession,\nOne Microsoft, diversity and inclusion, and making a difference.\n\nMicrosoft's 

In [47]:
len(chunks)

43

In [48]:
chunks[42].page_content

'============================================================\nEND OF MICROSOFT PUBLIC HR DATASET\n============================================================'

In [51]:
total_chars = sum(len(doc.page_content) for doc in chunks)

print("Chunks:", len(chunks))
print("Characters:", total_chars)
print("Approx tokens:", total_chars // 4)

Chunks: 43
Characters: 18113
Approx tokens: 4528


In [61]:
from langchain_community.embeddings import JinaEmbeddings
load_dotenv(override=True)

jina_api_key = os.getenv("JINA_API_KEY")

if not jina_api_key:
    raise ValueError("JINA_API_KEY is not set. Add it to your .env file or environment variables.")

embeddings_model = JinaEmbeddings(
    model_name="jina-embeddings-v5-text-small",
    jina_api_key= jina_api_key
)

print("EMB MODEL READY THE NAME IS ", embeddings_model.model_name)

EMB MODEL READY THE NAME IS  jina-embeddings-v5-text-small


In [62]:
from langchain_community.vectorstores import FAISS 

vector_store_new_1 = FAISS.from_documents(chunks , embeddings_model)

print("CHUNKS ARE STORED" , vector_store_new_1.index.ntotal)

CHUNKS ARE STORED 43


In [63]:
test_query = "tell me work ethics in the microsoft"

## SIMILARITY SEARCH 

top_matches = vector_store_new_1.similarity_search(test_query , k=2)
print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches,start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()

Query: tell me work ethics in the microsoft

--- Match 1 ---
Company:
Microsoft Corporation

Microsoft's mission is to empower every person and every organization on
the planet to achieve more.

Microsoft describes its culture around a growth mindset, customer obsession,
One Microsoft, diversity and inclusion, and making a difference.

Microsoft's Standards of Business Conduct / Trust Code emphasizes:
- Respect
- Integrity
- Accountability

SOURCE:
Microsoft Standards of Business Conduct
https://www.microsoft.com/en-us/legal/compliance/sbc/standards

--- Match 2 ---
SOURCE:
Microsoft Standards of Business Conduct
https://www.microsoft.com/en-us/legal/compliance/sbc/standards


15. TRUST CODE / STANDARDS OF BUSINESS CONDUCT

Microsoft's Standards of Business Conduct are also referred to as the
Trust Code.



In [64]:


retriever = vector_store_new_1.as_retriever(search_kwargs={"k": 3})  # returns top 3 relevant chunks

def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)



In [65]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature=0.5  # creativity 
)

llm.model_name

'openai/gpt-oss-120b'

In [66]:


tr = llm.invoke("Hey what is the leave policy")



In [67]:
tr

AIMessage(content='Sure thing!\u202fLeave policies can vary a lot depending on the country, the type of organization (public vs. private), the industry, and even the specific company’s own handbook. \n\nHere’s a quick rundown of the most common categories you’ll usually see:\n\n| **Leave Type** | **Typical Eligibility / How It Works** | **Typical Duration (varies widely)** |\n|----------------|----------------------------------------|--------------------------------------|\n| **Annual / Vacation Leave** | Full‑time employees (often after a probation period). Some places prorate based on months worked. | 10–30\u202fdays per year (often 2–4\u202fweeks). |\n| **Sick Leave** | Usually available from day\u202f1; may require a medical certificate after a certain number of days. | 5–15\u202fdays per year (some jurisdictions have “unlimited” sick leave). |\n| **Maternity / Parental Leave** | Pregnant employees (or new parents) – often protected by law. May be paid, partially paid, or unpaid. |

In [68]:
from langchain.agents import create_agent  

In [69]:
hr_assistant = create_agent(
    model = llm,
    tools=[search_hr_policy],
    system_prompt= """ 
    
    You are a friendly HR assistant working for Microsoft. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    """
)

print("HR assistant agent is ready to answer questions!")

HR assistant agent is ready to answer questions!


In [70]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [71]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "tell me which org you work for"
            }
        ]
    }
)

In [72]:
response

{'messages': [HumanMessage(content='tell me which org you work for', additional_kwargs={}, response_metadata={}, id='85fbe5b3-5448-4ab8-aee7-ec891e6304b0'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "tell me which org you work for". The assistant is a friendly HR assistant working for Microsoft. According to developer instruction, we must use search_hr_policy tool to look up facts before answering. However the question is about which organization the assistant works for, which is not about HR policy. The instruction says always use the search_hr_policy tool to look up facts before answering. If the answer isn\'t in the search results, say you don\'t know instead of guessing.\n\nThus we need to search HR policy for "org you work for" or "Microsoft". Probably the policy doc may contain organization name? Might not. We can attempt search: "Microsoft".', 'tool_calls': [{'id': 'fc_4e0f0176-9c58-49d9-a7a6-4b94f5440643', 'function': {'arguments': '{"quest

In [73]:
print(response["messages"][-1].content)

I’m a virtual HR assistant for **Microsoft**.


In [75]:
response['messages'][2].content

"company:\nMicrosoft\n\nsource_type:\nofficial_public\n\nregion:\nGlobal / United States\n\ncategory:\nBenefits / Leave / Conduct / Compliance / Legal / Compensation /\nWorkplace / Diversity\n\ndocument:\nName of source document\n\nsource_url:\nOfficial Microsoft URL\n\naccess_type:\nPublic information\n\n\n============================================================\n26. IMPORTANT REGION RULE\n============================================================\n\nCompany:\nMicrosoft Corporation\n\nMicrosoft's mission is to empower every person and every organization on\nthe planet to achieve more.\n\nMicrosoft describes its culture around a growth mindset, customer obsession,\nOne Microsoft, diversity and inclusion, and making a difference.\n\nMicrosoft's Standards of Business Conduct / Trust Code emphasizes:\n- Respect\n- Integrity\n- Accountability\n\nSOURCE:\nMicrosoft Standards of Business Conduct\nhttps://www.microsoft.com/en-us/legal/compliance/sbc/standards\n\nSOURCE:\nMicrosoft Caree